# 两阶段抽取 Demo

- 实体类字段: 候选定位 -> LLM 归一化
- 规则类字段: 句段召回 -> LLM 结构化输出
- 当前已接入通义千问兼容 OpenAI SDK 的调用方式, 无法调用时自动退回到本地 fallback

In [11]:
import json
import os
import re
from collections import defaultdict
from copy import deepcopy

import pandas as pd

with open("data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data["doc_id"])
print("paragraph count:", len(data.get("paragraphs", [])))
print("table count:", len(data.get("tables", [])))

2023年文化和旅游发展统计公报.md
paragraph count: 53
table count: 0


In [12]:
QWEN_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
QWEN_API_KEY = "sk-640a6d0c77a6454286e029fe74d8a47b"
QWEN_MODEL = "qwen-plus"

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

def build_qwen_client(api_key=None, base_url=QWEN_BASE_URL):
    api_key = api_key or QWEN_API_KEY
    if not api_key:
        return None
    if OpenAI is None:
        return None
    return OpenAI(api_key="sk-640a6d0c77a6454286e029fe74d8a47b", base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")

qwen_client = build_qwen_client()

print({
    "has_openai_sdk": OpenAI is not None,
    "has_api_key": bool(QWEN_API_KEY),
    "model": QWEN_MODEL,
    "base_url": QWEN_BASE_URL
})

{'has_openai_sdk': False, 'has_api_key': True, 'model': 'qwen-plus', 'base_url': 'https://dashscope.aliyuncs.com/compatible-mode/v1'}


当前 notebook 已直接写入通义千问 API Key 和模型名, 打开后可直接运行。

如果后续要换 key 或换模型, 直接修改 `llm-config` 单元里的 `QWEN_API_KEY` 和 `QWEN_MODEL`。

In [13]:
synonym_df = pd.read_excel("field_synonyms.xlsx")

synonyms_dict = {}
for _, row in synonym_df.iterrows():
    field = str(row["标准字段"]).strip()
    synonyms = [item.strip() for item in str(row["同义词"]).split("、") if item.strip()]
    synonyms_dict[field] = synonyms

reverse_dict = {}
for field, synonyms in synonyms_dict.items():
    for word in synonyms:
        reverse_dict[word] = field

ENTITY_FIELDS = [
    "项目名称", "项目负责人", "单位名称", "甲方", "乙方", "联系人", "联系电话",
    "项目编号", "合同编号", "日期", "金额"
]

ENTITY_RULE_PATTERNS = {
    "项目名称": [r"(?:项目名称|项目名|课题名称|合同名称|工程名称)[:：]\s*([^\n，。；;]{2,100})"],
    "项目负责人": [r"(?:项目负责人|负责人|课题负责人)[:：]\s*([^\s，。,；;：:\n]{2,30})"],
    "单位名称": [r"(?:单位名称|申报单位|所属单位|公司名称|机构名称)[:：]\s*([^\n，。；;]{2,100})"],
    "甲方": [r"(?:甲方|采购人|发包人|委托方)[:：]\s*([^\n，。；;]{2,100})"],
    "乙方": [r"(?:乙方|供应商|承包方|投标人)[:：]\s*([^\n，。；;]{2,100})"],
    "联系人": [r"(?:联系人|项目联系人)[:：]\s*([^\s，。,；;：:\n]{2,30})"],
    "联系电话": [r"(?:联系电话|联系人电话|手机号|手机|电话)[:：]?\s*([0-9\-+() ]{7,20})"],
    "项目编号": [r"(?:项目编号|项目编码|项目号|立项编号|招标编号)[:：]\s*([A-Za-z0-9\-_/]+)"],
    "合同编号": [r"(?:合同编号|合同编码|合同号|协议编号)[:：]\s*([A-Za-z0-9\-_/]+)"],
    "日期": [r"(?:发布时间|签订日期|时间|日期)[:：]?\s*((?:19|20)\d{2}[年\-/]\d{1,2}[月\-/]\d{1,2}日?)"],
    "金额": [r"(?:金额|总金额|合同金额|预算金额|中标金额)[:：]?\s*([0-9][0-9,\.]*\s*(?:元|万元|亿元))"]
}

RULE_FIELD_CONFIG = {
    "适用条件": ["符合", "满足", "具备", "条件", "要求", "适用"],
    "禁止约束": ["不得", "禁止", "严禁", "不可", "不应"],
    "办理时限": ["工作日", "截止", "期限", "时限", "之前", "之内"],
    "申报材料": ["提交", "提供", "材料", "证明", "附件", "申请表"]
}

list(reverse_dict.items())[:10]

[('项目名称', '项目名称'),
 ('项目名', '项目名称'),
 ('课题名称', '项目名称'),
 ('合同名称', '项目名称'),
 ('工程名称', '项目名称'),
 ('标段名称', '项目名称'),
 ('采购项目', '项目名称'),
 ('项目', '项目名称'),
 ('项目编号', '项目编号'),
 ('项目编码', '项目编号')]

In [14]:
def split_sentences(paragraph):
    paragraph = paragraph.strip()
    if not paragraph:
        return []
    parts = re.split(r"(?<=[。！？；;])", paragraph)
    return [item.strip() for item in parts if item.strip()]

def clamp_score(value):
    return round(max(0.0, min(1.0, value)), 3)

def keyword_confidence(trigger):
    if len(trigger) >= 4:
        return 0.82
    if len(trigger) >= 2:
        return 0.75
    return 0.66

def extract_json_text(raw_text):
    raw_text = raw_text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", raw_text, re.S)
    if fenced:
        return fenced.group(1).strip()
    return raw_text

def safe_load_json(raw_text, default_value):
    try:
        return json.loads(extract_json_text(raw_text))
    except Exception:
        return default_value

def chat_json(messages, default_value, model=QWEN_MODEL, temperature=0.1):
    if qwen_client is None:
        return default_value
    try:
        response = qwen_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        content = response.choices[0].message.content
        return safe_load_json(content, default_value)
    except Exception:
        return default_value

def pseudo_ner_entity_candidates(paragraphs):
    candidates = defaultdict(list)
    org_pattern = r"([\u4e00-\u9fa5A-Za-z0-9]{2,40}(?:公司|集团|大学|学院|研究院|研究所|委员会|人民政府|文化和旅游部|国家文物局))"
    project_pattern = r"(?:《([^》]{2,60})》|“([^”]{2,60})”|项目名称[:：]\s*([^\n，。；;]{2,80}))"

    for para_id, para in enumerate(paragraphs):
        for match in re.finditer(org_pattern, para):
            value = match.group(1).strip()
            candidates["单位名称"].append({
                "field": "单位名称",
                "candidate_text": value,
                "snippet": para,
                "para_id": para_id,
                "source": "ner",
                "trigger": "org_pattern",
                "confidence": 0.68
            })

        for match in re.finditer(project_pattern, para):
            value = next((group for group in match.groups() if group), None)
            if value:
                candidates["项目名称"].append({
                    "field": "项目名称",
                    "candidate_text": value.strip(),
                    "snippet": para,
                    "para_id": para_id,
                    "source": "ner",
                    "trigger": "project_pattern",
                    "confidence": 0.64
                })

    return dict(candidates)

In [15]:
def locate_entity_candidates(paragraphs, reverse_dict, rule_patterns, target_fields=None, window=100):
    target_fields = set(target_fields or rule_patterns.keys())
    candidates = defaultdict(list)

    for para_id, para in enumerate(paragraphs):
        para = para.strip()
        if not para:
            continue

        for field, patterns in rule_patterns.items():
            if field not in target_fields:
                continue
            for pattern in patterns:
                for match in re.finditer(pattern, para):
                    value = match.group(1).strip()
                    candidates[field].append({
                        "field": field,
                        "candidate_text": value,
                        "snippet": para,
                        "para_id": para_id,
                        "source": "rule",
                        "trigger": pattern,
                        "confidence": 0.93
                    })

        for trigger, field in reverse_dict.items():
            if field not in target_fields or trigger not in para:
                continue

            hit_pos = para.find(trigger)
            start = max(0, hit_pos - 24)
            end = min(len(para), hit_pos + len(trigger) + window)

            candidates[field].append({
                "field": field,
                "candidate_text": para[start:end].strip(),
                "snippet": para,
                "para_id": para_id,
                "source": "keyword",
                "trigger": trigger,
                "confidence": keyword_confidence(trigger)
            })

    ner_candidates = pseudo_ner_entity_candidates(paragraphs)
    for field, items in ner_candidates.items():
        if field in target_fields:
            candidates[field].extend(items)

    return dict(candidates)

entity_candidates = locate_entity_candidates(
    data["paragraphs"],
    reverse_dict,
    ENTITY_RULE_PATTERNS,
    target_fields=ENTITY_FIELDS
)

{field: items[:2] for field, items in entity_candidates.items() if items}

{'日期': [{'field': '日期',
   'candidate_text': '2024年08月30日',
   'snippet': '中华人民共和国文化和旅游部  \n发布时间：2024年08月30日',
   'para_id': 1,
   'source': 'rule',
   'trigger': '(?:发布时间|签订日期|时间|日期)[:：]?\\s*((?:19|20)\\d{2}[年\\-/]\\d{1,2}[月\\-/]\\d{1,2}日?)',
   'confidence': 0.93},
  {'field': '日期',
   'candidate_text': '中华人民共和国文化和旅游部  \n发布时间：2024年08月30日',
   'snippet': '中华人民共和国文化和旅游部  \n发布时间：2024年08月30日',
   'para_id': 1,
   'source': 'keyword',
   'trigger': '时间',
   'confidence': 0.75}],
 '单位名称': [{'field': '单位名称',
   'candidate_text': '2023年末，纳入统计报送的全国各类文化和旅游单位30.4万个。其中，各级文化和旅游部门所属单位6.6万个，从业人员72.9万人。',
   'snippet': '2023年末，纳入统计报送的全国各类文化和旅游单位30.4万个。其中，各级文化和旅游部门所属单位6.6万个，从业人员72.9万人。',
   'para_id': 3,
   'source': 'keyword',
   'trigger': '单位',
   'confidence': 0.75},
  {'field': '单位名称',
   'candidate_text': '指南，加强水书、甲骨文保护，组织全国古籍重点保护单位复核抽查，推进古籍数字化整理、中华古籍智慧化服务平台建设，古籍保护利用更加科学规范。',
   'snippet': '2023年，文化和旅游部、国家文物局持续推进中华文明探源工程和“考古中国”重大项目。水下考古取得重大突破，深海考古迈向世界先进水平，边疆考古不断加强。启动第四次全国文物普查，强化国家考古遗址公园保护建设，加强大

In [16]:
def build_entity_llm_prompt(field, candidates):
    return {
        "task": "你是信息抽取器。请从候选片段中选择最可信的一个值，并归一化为标准 JSON。只返回 JSON，不要解释。",
        "field": field,
        "expected_output": {
            "field": field,
            "value": "string or null",
            "confidence": "0~1",
            "source": "rule|keyword|ner|llm",
            "reason": "short explanation",
            "evidence": {
                "para_id": "int",
                "snippet": "string",
                "trigger": "string"
            }
        },
        "candidates": candidates
    }

def fallback_entity_llm_fn(field, candidates):
    ranked = sorted(candidates, key=lambda x: x["confidence"], reverse=True)
    best = ranked[0]
    return {
        "field": field,
        "value": best["candidate_text"],
        "confidence": clamp_score(best["confidence"]),
        "source": best["source"],
        "reason": "fallback: choose highest-confidence candidate",
        "evidence": {
            "para_id": best["para_id"],
            "snippet": best["snippet"],
            "trigger": best["trigger"]
        }
    }

def qwen_entity_llm_fn(field, candidates):
    fallback = fallback_entity_llm_fn(field, candidates)
    prompt = build_entity_llm_prompt(field, candidates)
    messages = [
        {"role": "system", "content": "你是一个严谨的信息抽取助手。输出必须是合法 JSON。"},
        {"role": "user", "content": json.dumps(prompt, ensure_ascii=False, indent=2)}
    ]
    result = chat_json(messages, fallback)
    if not isinstance(result, dict):
        return fallback
    result.setdefault("field", field)
    result.setdefault("value", fallback["value"])
    result["confidence"] = clamp_score(float(result.get("confidence", fallback["confidence"])))
    result.setdefault("source", "llm")
    result.setdefault("reason", "qwen completion")
    result.setdefault("evidence", fallback["evidence"])
    return result

def normalize_entity_candidates(entity_candidates, llm_fn=None):
    llm_fn = llm_fn or qwen_entity_llm_fn
    entity_result = {}
    entity_prompts = {}

    for field, candidates in entity_candidates.items():
        if not candidates:
            continue
        entity_prompts[field] = build_entity_llm_prompt(field, candidates)
        entity_result[field] = llm_fn(field, deepcopy(candidates))

    return entity_result, entity_prompts

entity_result, entity_prompts = normalize_entity_candidates(entity_candidates)
entity_result

{'日期': {'field': '日期',
  'value': '2024年08月30日',
  'confidence': 0.93,
  'source': 'rule',
  'reason': 'fallback: choose highest-confidence candidate',
  'evidence': {'para_id': 1,
   'snippet': '中华人民共和国文化和旅游部  \n发布时间：2024年08月30日',
   'trigger': '(?:发布时间|签订日期|时间|日期)[:：]?\\s*((?:19|20)\\d{2}[年\\-/]\\d{1,2}[月\\-/]\\d{1,2}日?)'}},
 '单位名称': {'field': '单位名称',
  'value': '2023年末，纳入统计报送的全国各类文化和旅游单位30.4万个。其中，各级文化和旅游部门所属单位6.6万个，从业人员72.9万人。',
  'confidence': 0.75,
  'source': 'keyword',
  'reason': 'fallback: choose highest-confidence candidate',
  'evidence': {'para_id': 3,
   'snippet': '2023年末，纳入统计报送的全国各类文化和旅游单位30.4万个。其中，各级文化和旅游部门所属单位6.6万个，从业人员72.9万人。',
   'trigger': '单位'}},
 '项目名称': {'field': '项目名称',
  'value': '文化空间建设。推进全国智慧图书馆体系和公共文化云项目建设。进一步推动落实基本公共文化服务标准。开展第七次全国县级以上公共图书馆评估定级工作，共评定一级公共图书馆1302家、二级公共图书馆680家、三级公共图书馆741家。开展公共图书馆、文化馆服务宣传周',
  'confidence': 0.75,
  'source': 'keyword',
  'reason': 'fallback: choose highest-confidence candidate',
  'evidence': {'para_id': 10,
   'snippet': '2023年

In [17]:
def locate_rule_candidates(paragraphs, rule_field_config):
    results = defaultdict(list)

    for para_id, para in enumerate(paragraphs):
        sentences = split_sentences(para)
        for sent_id, sent in enumerate(sentences):
            for rule_field, triggers in rule_field_config.items():
                matched = [trigger for trigger in triggers if trigger in sent]
                if not matched:
                    continue

                score = clamp_score(0.62 + 0.08 * min(len(matched), 3))
                results[rule_field].append({
                    "field": rule_field,
                    "para_id": para_id,
                    "sentence_id": sent_id,
                    "candidate_text": sent,
                    "snippet": para,
                    "matched_triggers": matched,
                    "source": "rule_recall",
                    "confidence": score
                })

    return dict(results)

rule_candidates = locate_rule_candidates(data["paragraphs"], RULE_FIELD_CONFIG)
{field: items[:2] for field, items in rule_candidates.items() if items}

{'适用条件': [{'field': '适用条件',
   'para_id': 2,
   'sentence_id': 0,
   'candidate_text': '2023年，在以习近平同志为核心的党中央坚强领导下，全国文化和旅游系统坚持以习近平新时代中国特色社会主义思想为指导，认真贯彻落实党的二十大精神，深入学习贯彻习近平文化思想，坚持稳中求进工作总基调，完整、准确、全面贯彻新发展理念，着力推动高质量发展，统筹好发展和安全，坚持以社会主义核心价值观为引领，以满足人民文化需求和增强人民精神力量为着力点，努力创作优秀文化作品、提供优秀文化产品和优质旅游产品，推动文化和旅游行业实现快速恢复发展，有力发挥了稳定增长、促进消费、扩大就业、惠及民生的重要作用。',
   'snippet': '2023年，在以习近平同志为核心的党中央坚强领导下，全国文化和旅游系统坚持以习近平新时代中国特色社会主义思想为指导，认真贯彻落实党的二十大精神，深入学习贯彻习近平文化思想，坚持稳中求进工作总基调，完整、准确、全面贯彻新发展理念，着力推动高质量发展，统筹好发展和安全，坚持以社会主义核心价值观为引领，以满足人民文化需求和增强人民精神力量为着力点，努力创作优秀文化作品、提供优秀文化产品和优质旅游产品，推动文化和旅游行业实现快速恢复发展，有力发挥了稳定增长、促进消费、扩大就业、惠及民生的重要作用。',
   'matched_triggers': ['满足'],
   'source': 'rule_recall',
   'confidence': 0.7},
  {'field': '适用条件',
   'para_id': 38,
   'sentence_id': 2,
   'candidate_text': '印发《文化和旅游标准化工作管理办法》，进一步规范文化和旅游标准化工作，推动出台国家标准5项，批准发布行业标准15项、立项28项，推进2项国际标准立项，覆盖旅游、剧场、图书馆、文化馆、非遗、网络文化、动漫等领域，涉及乡村旅游、智慧旅游等业态评价，旅游景区、饭店、演艺新空间、网络表演等管理服务要求，演出设备、动漫平台等技术规范，不断提升管理和服务水平，为行业高质量发展奠定了基础。',
   'snippet': '2023年，文化和旅游部持续推进实施国家文化数字化战略

In [18]:
def build_rule_llm_prompt(field, candidates):
    return {
        "task": "你是规则抽取器。请识别规则文本中的 condition、subject、action、constraint，并返回 JSON 数组。只返回 JSON，不要解释。",
        "field": field,
        "expected_output": [{
            "field": field,
            "condition": "string or null",
            "subject": "string or null",
            "action": "string or null",
            "constraint": "string or null",
            "confidence": "0~1",
            "source": "rule_recall|llm",
            "evidence": {
                "para_id": "int",
                "sentence_id": "int",
                "snippet": "string",
                "triggers": []
            }
        }],
        "candidates": candidates
    }

def fallback_rule_llm_fn(field, candidates):
    outputs = []
    for item in sorted(candidates, key=lambda x: x["confidence"], reverse=True):
        sentence = item["candidate_text"]
        outputs.append({
            "field": field,
            "condition": sentence if field == "适用条件" else None,
            "subject": None,
            "action": sentence if field in {"申报材料", "办理时限"} else None,
            "constraint": sentence if field == "禁止约束" else None,
            "confidence": clamp_score(item["confidence"]),
            "source": item["source"],
            "evidence": {
                "para_id": item["para_id"],
                "sentence_id": item["sentence_id"],
                "snippet": item["candidate_text"],
                "triggers": item["matched_triggers"]
            }
        })
    return outputs

def qwen_rule_llm_fn(field, candidates):
    fallback = fallback_rule_llm_fn(field, candidates)
    prompt = build_rule_llm_prompt(field, candidates)
    messages = [
        {"role": "system", "content": "你是一个严谨的规则抽取助手。输出必须是合法 JSON 数组。"},
        {"role": "user", "content": json.dumps(prompt, ensure_ascii=False, indent=2)}
    ]
    result = chat_json(messages, fallback)
    if not isinstance(result, list):
        return fallback
    normalized = []
    for idx, item in enumerate(result):
        base = fallback[min(idx, len(fallback) - 1)] if fallback else {
            "field": field,
            "condition": None,
            "subject": None,
            "action": None,
            "constraint": None,
            "confidence": 0.5,
            "source": "llm",
            "evidence": {}
        }
        if not isinstance(item, dict):
            normalized.append(base)
            continue
        item.setdefault("field", field)
        item["confidence"] = clamp_score(float(item.get("confidence", base["confidence"])))
        item.setdefault("source", "llm")
        item.setdefault("evidence", base["evidence"])
        item.setdefault("condition", base.get("condition"))
        item.setdefault("subject", base.get("subject"))
        item.setdefault("action", base.get("action"))
        item.setdefault("constraint", base.get("constraint"))
        normalized.append(item)
    return normalized

def structure_rule_candidates(rule_candidates, llm_fn=None):
    llm_fn = llm_fn or qwen_rule_llm_fn
    rule_result = {}
    rule_prompts = {}

    for field, candidates in rule_candidates.items():
        if not candidates:
            continue
        rule_prompts[field] = build_rule_llm_prompt(field, candidates)
        rule_result[field] = llm_fn(field, deepcopy(candidates))

    return rule_result, rule_prompts

rule_result, rule_prompts = structure_rule_candidates(rule_candidates)
rule_result

{'适用条件': [{'field': '适用条件',
   'condition': '2023年，在以习近平同志为核心的党中央坚强领导下，全国文化和旅游系统坚持以习近平新时代中国特色社会主义思想为指导，认真贯彻落实党的二十大精神，深入学习贯彻习近平文化思想，坚持稳中求进工作总基调，完整、准确、全面贯彻新发展理念，着力推动高质量发展，统筹好发展和安全，坚持以社会主义核心价值观为引领，以满足人民文化需求和增强人民精神力量为着力点，努力创作优秀文化作品、提供优秀文化产品和优质旅游产品，推动文化和旅游行业实现快速恢复发展，有力发挥了稳定增长、促进消费、扩大就业、惠及民生的重要作用。',
   'subject': None,
   'action': None,
   'constraint': None,
   'confidence': 0.7,
   'source': 'rule_recall',
   'evidence': {'para_id': 2,
    'sentence_id': 0,
    'snippet': '2023年，在以习近平同志为核心的党中央坚强领导下，全国文化和旅游系统坚持以习近平新时代中国特色社会主义思想为指导，认真贯彻落实党的二十大精神，深入学习贯彻习近平文化思想，坚持稳中求进工作总基调，完整、准确、全面贯彻新发展理念，着力推动高质量发展，统筹好发展和安全，坚持以社会主义核心价值观为引领，以满足人民文化需求和增强人民精神力量为着力点，努力创作优秀文化作品、提供优秀文化产品和优质旅游产品，推动文化和旅游行业实现快速恢复发展，有力发挥了稳定增长、促进消费、扩大就业、惠及民生的重要作用。',
    'triggers': ['满足']}},
  {'field': '适用条件',
   'condition': '印发《文化和旅游标准化工作管理办法》，进一步规范文化和旅游标准化工作，推动出台国家标准5项，批准发布行业标准15项、立项28项，推进2项国际标准立项，覆盖旅游、剧场、图书馆、文化馆、非遗、网络文化、动漫等领域，涉及乡村旅游、智慧旅游等业态评价，旅游景区、饭店、演艺新空间、网络表演等管理服务要求，演出设备、动漫平台等技术规范，不断提升管理和服务水平，为行业高质量发展奠定了基础。',
   'subject': None,


In [19]:
def build_db_records(doc_id, entity_result, rule_result):
    records = []

    for field, payload in entity_result.items():
        records.append({
            "doc_id": doc_id,
            "field": field,
            "record_type": "entity",
            "value": payload["value"],
            "confidence": payload["confidence"],
            "source": payload["source"],
            "evidence": payload["evidence"]
        })

    for field, items in rule_result.items():
        for payload in items:
            records.append({
                "doc_id": doc_id,
                "field": field,
                "record_type": "rule",
                "value": {
                    "condition": payload["condition"],
                    "subject": payload["subject"],
                    "action": payload["action"],
                    "constraint": payload["constraint"]
                },
                "confidence": payload["confidence"],
                "source": payload["source"],
                "evidence": payload["evidence"]
            })

    return records

def extraction_pipeline(data, reverse_dict, entity_rule_patterns, rule_field_config, entity_llm_fn=None, rule_llm_fn=None):
    entity_candidates = locate_entity_candidates(
        data["paragraphs"],
        reverse_dict,
        entity_rule_patterns,
        target_fields=ENTITY_FIELDS
    )
    entity_result, entity_prompts = normalize_entity_candidates(entity_candidates, llm_fn=entity_llm_fn)

    rule_candidates = locate_rule_candidates(data["paragraphs"], rule_field_config)
    rule_result, rule_prompts = structure_rule_candidates(rule_candidates, llm_fn=rule_llm_fn)

    db_records = build_db_records(data["doc_id"], entity_result, rule_result)

    return {
        "doc_id": data["doc_id"],
        "entity_candidates": entity_candidates,
        "entity_result": entity_result,
        "rule_candidates": rule_candidates,
        "rule_result": rule_result,
        "entity_prompts": entity_prompts,
        "rule_prompts": rule_prompts,
        "db_records": db_records
    }

pipeline_result = extraction_pipeline(
    data,
    reverse_dict,
    ENTITY_RULE_PATTERNS,
    RULE_FIELD_CONFIG
)

pipeline_result.keys()

dict_keys(['doc_id', 'entity_candidates', 'entity_result', 'rule_candidates', 'rule_result', 'entity_prompts', 'rule_prompts', 'db_records'])

In [20]:
preview = {
    "doc_id": pipeline_result["doc_id"],
    "entity_result": pipeline_result["entity_result"],
    "rule_result": {k: v[:2] for k, v in pipeline_result["rule_result"].items()},
    "db_records_preview": pipeline_result["db_records"][:5]
}

print(json.dumps(preview, ensure_ascii=False, indent=2))

{
  "doc_id": "2023年文化和旅游发展统计公报.md",
  "entity_result": {
    "日期": {
      "field": "日期",
      "value": "2024年08月30日",
      "confidence": 0.93,
      "source": "rule",
      "reason": "fallback: choose highest-confidence candidate",
      "evidence": {
        "para_id": 1,
        "snippet": "中华人民共和国文化和旅游部  \n发布时间：2024年08月30日",
        "trigger": "(?:发布时间|签订日期|时间|日期)[:：]?\\s*((?:19|20)\\d{2}[年\\-/]\\d{1,2}[月\\-/]\\d{1,2}日?)"
      }
    },
    "单位名称": {
      "field": "单位名称",
      "value": "2023年末，纳入统计报送的全国各类文化和旅游单位30.4万个。其中，各级文化和旅游部门所属单位6.6万个，从业人员72.9万人。",
      "confidence": 0.75,
      "source": "keyword",
      "reason": "fallback: choose highest-confidence candidate",
      "evidence": {
        "para_id": 3,
        "snippet": "2023年末，纳入统计报送的全国各类文化和旅游单位30.4万个。其中，各级文化和旅游部门所属单位6.6万个，从业人员72.9万人。",
        "trigger": "单位"
      }
    },
    "项目名称": {
      "field": "项目名称",
      "value": "文化空间建设。推进全国智慧图书馆体系和公共文化云项目建设。进一步推动落实基本公共文化服务标准。开展第七次全国县级以上公共图书馆评估定级工作，共评定一级公共图书馆1302家、二级公共图

## 说明

- 已接入 `base_url=https://dashscope.aliyuncs.com/compatible-mode/v1`
- 默认模型是 `qwen-plus`, 可通过环境变量 `QWEN_MODEL` 覆盖
- 如果本机缺少 `openai` SDK 或没有设置 `DASHSCOPE_API_KEY`, notebook 仍可运行, 但会走 fallback 结果

In [21]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-640a6d0c77a6454286e029fe74d8a47b",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "你是谁？"},
    ],
)

print(completion.model_dump_json())

{"id":"chatcmpl-0a51a37c-dfe4-95ff-8ff9-5de772fbc31a","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊","refusal":null,"role":"assistant","annotations":null,"audio":null,"function_call":null,"tool_calls":null}}],"created":1773405304,"model":"qwen-plus","object":"chat.completion","service_tier":null,"system_fingerprint":null,"usage":{"completion_tokens":66,"prompt_tokens":22,"total_tokens":88,"completion_tokens_details":null,"prompt_tokens_details":{"audio_tokens":null,"cached_tokens":0}}}
